# HORM E/F versus E/F/H screening

Open this notebook from GitHub in a fresh GPU runtime. Before running any cell, use the left **Files** panel to upload `oa_audit_01_generation_8x8_r2_j2.tar.gz` into `/content`. This avoids Colab's JavaScript upload helper.

In [ ]:
from pathlib import Path

INPUT_BUNDLE = Path('/content/oa_audit_01_generation_8x8_r2_j2.tar.gz')
assert INPUT_BUNDLE.is_file(), 'Upload the stage-01 archive to /content with the left Files panel first.'
print(f'Input bundle: {INPUT_BUNDLE.stat().st_size / 1024**2:.2f} MiB')

In [ ]:
from pathlib import Path
import os
import tarfile

input_bundle = INPUT_BUNDLE
OUTPUT_ROOT = Path('/content/oa_audit_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
output_root_resolved = OUTPUT_ROOT.resolve()
with tarfile.open(input_bundle, 'r:gz') as archive:
    for member in archive.getmembers():
        target = (OUTPUT_ROOT / member.name).resolve()
        assert target == output_root_resolved or output_root_resolved in target.parents
        assert member.isfile() or member.isdir(), f'Unsupported archive entry: {member.name}'
    archive.extractall(OUTPUT_ROOT)

REPOSITORY_URL = 'https://github.com/jiaxi98/OAReactDiff.git'
REPOSITORY_REF = 'agent/oa-failure-audit'
REPO = Path('/content/OAReactDiff')
MAMBA = '/usr/local/bin/micromamba'
ENV_PREFIX = Path('/content/micromamba/envs/oa-horm')
os.environ['LD_LIBRARY_PATH'] = f"{ENV_PREFIX}/lib:" + os.environ.get('LD_LIBRARY_PATH', '')
MANIFEST = OUTPUT_ROOT / 'generation_8x8_r2_j2/candidate_manifest.csv'
SCREEN_ROOT = OUTPUT_ROOT / 'horm_screen_generation_8x8_r2_j2'
MODEL_ROOT = Path('/content/models/HORM')
assert MANIFEST.is_file(), f'The uploaded stage-01 bundle is missing {MANIFEST}'
SCREEN_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
if not (REPO / '.git').is_dir():
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 --branch {REPOSITORY_REF} {REPOSITORY_URL} {REPO}
else:
    !git -C {REPO} fetch --depth 1 origin {REPOSITORY_REF}
    !git -C {REPO} checkout --detach FETCH_HEAD
!git -C {REPO} rev-parse HEAD
!cd {REPO} && bash experiments/oa_failure_audit/setup_colab_horm.sh

In [ ]:
REVISION = 'a582ca8f0bfb9c50c6e384e4ef1db9b0a8dd1dd4'
EF = MODEL_ROOT / 'left_orig.ckpt'
EFH = MODEL_ROOT / 'left.ckpt'
if not EF.is_file():
    !curl -L --fail --output {EF} https://huggingface.co/yhong55/HORM/resolve/{REVISION}/left_orig.ckpt
if not EFH.is_file():
    !curl -L --fail --output {EFH} https://huggingface.co/yhong55/HORM/resolve/{REVISION}/left.ckpt
!echo '1c286d36152781d1923cf6ab778d2f5227cf8bb626e07604dc8b86f15a6ac6fa  '{EF} | sha256sum --check
!echo '55b1f2d21897ad4f7870986397ed981185989fc947f08aff172c65cd41a1f2a0  '{EFH} | sha256sum --check

In [ ]:
import csv

HORM_REPO = Path('/content/HORM')
EF_RESULT = SCREEN_ROOT / 'horm_left_ef.csv'
EFH_RESULT = SCREEN_ROOT / 'horm_left_efh.csv'
SMOKE_ROOT = SCREEN_ROOT / 'smoke'
SMOKE_EF = SMOKE_ROOT / 'horm_left_ef_one.csv'
SMOKE_EFH = SMOKE_ROOT / 'horm_left_efh_one.csv'
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/screen_horm.py \
    --manifest {MANIFEST} --horm-repo {HORM_REPO} --checkpoint {EF} \
    --label horm_left_ef_smoke --output {SMOKE_EF} --device cuda --max-candidates 1 --resume
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/screen_horm.py \
    --manifest {MANIFEST} --horm-repo {HORM_REPO} --checkpoint {EFH} \
    --label horm_left_efh_smoke --output {SMOKE_EFH} --device cuda --max-candidates 1 --resume

required_smoke_fields = [
    'energy_ev', 'force_rms_ev_per_angstrom', 'lowest_frequency_cm',
    'second_frequency_cm', 'hvp_calls', 'wall_seconds',
]
for smoke_path in (SMOKE_EF, SMOKE_EFH):
    with smoke_path.open(newline='') as handle:
        smoke_rows = list(csv.DictReader(handle))
    assert len(smoke_rows) == 1, f'Expected one smoke-test row in {smoke_path}'
    assert not smoke_rows[0]['error'], smoke_rows[0]['error']
    assert all(smoke_rows[0][field] for field in required_smoke_fields), smoke_rows[0]
    print(smoke_path.name, 'passed in', smoke_rows[0]['wall_seconds'], 'seconds')
print('Both checkpoints passed. Run the next cell for the full 64-candidate screens.')

In [ ]:
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/screen_horm.py \
    --manifest {MANIFEST} --horm-repo {HORM_REPO} --checkpoint {EF} \
    --label horm_left_ef --output {EF_RESULT} --device cuda --resume
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/screen_horm.py \
    --manifest {MANIFEST} --horm-repo {HORM_REPO} --checkpoint {EFH} \
    --label horm_left_efh --output {EFH_RESULT} --device cuda --resume

for result_path in (EF_RESULT, EFH_RESULT):
    with result_path.open(newline='') as handle:
        result_rows = list(csv.DictReader(handle))
    failed_rows = [row for row in result_rows if row['error']]
    assert len(result_rows) == 64, f'Expected 64 rows in {result_path}, found {len(result_rows)}'
    assert not failed_rows, f'{result_path.name} has {len(failed_rows)} failed rows: {failed_rows[:3]}'
    print(result_path.name, 'completed 64/64 candidates')

In [ ]:
ENRICHED = SCREEN_ROOT / 'candidate_manifest_screened.csv'
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/merge_screening.py \
    --manifest {MANIFEST} --screen horm_left_ef={EF_RESULT} \
    --screen horm_left_efh={EFH_RESULT} --output {ENRICHED} --overwrite
DFT_SUBSET = SCREEN_ROOT / 'dft_subset.csv'
!cd {REPO} && {MAMBA} run -p {ENV_PREFIX} python experiments/oa_failure_audit/select_dft_subset.py \
    --manifest {ENRICHED} --output {DFT_SUBSET} --budget 96 --max-per-reaction 2 --overwrite

The 8 × 8 pilot can contribute at most 16 DFT cases under the two-per-reaction cap. Generate at least 25–50 reactions before constructing the intended 50–200 case DFT subset.

## Package the screening results for local download

This cell packages the stage-01 structures, HORM results, and DFT worklist. After it finishes, use the left **Files** panel to download the archive for stage 03 and local analysis.

In [ ]:
import hashlib
import shutil

bundle_path = Path(shutil.make_archive(
    '/content/oa_audit_02_horm_screen_generation_8x8_r2_j2',
    'gztar',
    root_dir=OUTPUT_ROOT,
    base_dir='.',
))
digest = hashlib.sha256()
with bundle_path.open('rb') as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b''):
        digest.update(chunk)
print(f'{bundle_path.name}: {bundle_path.stat().st_size / 1024**2:.2f} MiB')
print(f'sha256: {digest.hexdigest()}')
print(f'Bundle ready at {bundle_path}')
print('Open Files on the left, refresh, then right-click the archive and choose Download.')